# TARGET 4 - Dynamic Route ETA Model

**Goal:** Combine `routes_history` + `weather` + `traffic` + `tir_shipments` (departure_time, route_difficulty) to predict the expected delivery duration (minutes).

**Metric:** MAE (minutes), RMSE (minutes), R2
**Baseline:** simple `distance_km / avg_speed` ETA
**Models compared:** 6 different regressors, best one is tuned with full hyperparameter search
**Final layer:** LLM-generated natural-language ETA brief for dispatcher + customer

| File | Columns | Role |
|---|---|---|
| routes_history.parquet | distance_km, duration_minutes, fuel_used | Historical base ETA + TARGET |
| tir_shipments.parquet | departure_time, route_difficulty, actual_load_ton | Trip conditions |
| weather.parquet | rainfall_mm, wind_speed | Weather delay factor |
| traffic.parquet | congestion_level | Road conditions |


## 1. Libraries and data loading

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
import joblib
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)


routes  = pd.read_parquet('routes_history.parquet')
tir     = pd.read_parquet('tir_shipments.parquet')
traffic = pd.read_parquet('traffic.parquet')
weather = pd.read_parquet('weather.parquet')

print('routes_history :', routes.shape)
print('tir_shipments  :', tir.shape)
print('traffic        :', traffic.shape)
print('weather        :', weather.shape)


routes_history : (3000, 7)
tir_shipments  : (16000, 17)
traffic        : (30000, 3)
weather        : (23570, 5)


In [2]:

routes.head(3)


,route_id,vehicle_id,start_location,end_location,distance_km,duration_minutes,fuel_used
0,RT30000,VH1081,Qazakh,Khachmaz,681.9,653.6,142.07
1,RT30001,VH1031,Lankaran,Ganja,726.0,701.0,79.32
2,RT30002,VH1060,Yevlakh,Lankaran,791.4,982.8,67.71


In [3]:

tir.head(3)


,shipment_id,route_id,origin_hub,destination_hub,departure_date,departure_time,actual_load_ton,capacity_ton,utilization_rate,num_packages,is_delayed,delay_minutes,is_spot_rental,driver_id,route_difficulty,associated_order_ids,pricing_confidence
0,SH40000,RT32459,HUB_SHEKI,HUB_KHANKENDI,2025-01-16,23:30:00,12.506,13.84,0.903613,9,0,0.0,0,DR648,flat,OR946151|OR930661|OR902341|OR946491|OR901044,market_estimate
1,SH40001,RT30672,HUB_NAKHCHIVAN,HUB_KHACHMAZ,2025-09-14,22:30:00,11.944,12.74,0.937520,10,0,0.0,0,DR573,flat,OR925651|OR923180|OR935502|OR935463|OR929429,market_estimate
2,SH40002,RT31922,HUB_KHANKENDI,HUB_KALBAJAR,2021-10-31,22:30:00,15.225,16.33,0.932333,9,0,0.0,1,DR612,flat,OR933071|OR932503,market_estimate


## 2. Schema check

`tir_shipments.route_id` matches `routes_history.route_id` (TARGET = `duration_minutes`).
`origin_hub` (`HUB_XXX`) needs to be mapped to `weather.region` / `traffic.region` (`XXX`) by stripping the prefix.


In [4]:

print('route_id overlap (tir -> routes):', tir['route_id'].isin(routes['route_id']).mean().round(4))
print('region names (weather):', sorted(weather['region'].unique()))
print('hub names (tir):', sorted(tir['origin_hub'].unique()))


route_id overlap (tir -> routes): 0.7484
region names (weather): ['Absheron', 'Ganja', 'Kalbajar', 'Khachmaz', 'Khankendi', 'Lankaran', 'Nakhchivan', 'Qazakh', 'Sheki', 'Yevlakh']
hub names (tir): ['HUB_ABSHERON', 'HUB_GANJA', 'HUB_KALBAJAR', 'HUB_KHACHMAZ', 'HUB_KHANKENDI', 'HUB_LANKARAN', 'HUB_NAKHCHIVAN', 'HUB_QAZAKH', 'HUB_SHEKI', 'HUB_YEVLAKH']


## 3. Building the dataset (merge)

**3.1 — `tir_shipments` + `routes_history`** (inner join on `route_id`) → adds the TARGET (`duration_minutes`).

In [5]:

df = tir.merge(
    routes[['route_id', 'start_location', 'end_location', 'distance_km', 'duration_minutes', 'fuel_used']],
    on='route_id', how='inner'
)
df = df.reset_index(drop=True)
print('Merged shipment count:', df.shape)
df.head(3)


Merged shipment count: (11975, 22)


,shipment_id,route_id,origin_hub,destination_hub,departure_date,departure_time,actual_load_ton,capacity_ton,utilization_rate,num_packages,is_delayed,delay_minutes,is_spot_rental,driver_id,route_difficulty,associated_order_ids,pricing_confidence,start_location,end_location,distance_km,duration_minutes,fuel_used
0,SH40000,RT32459,HUB_SHEKI,HUB_KHANKENDI,2025-01-16,23:30:00,12.506,13.84,0.903613,9,0,0.0,0,DR648,flat,OR946151|OR930661|OR902341|OR946491|OR901044,market_estimate,Nakhchivan,Sheki,583.9,605.9,110.13
1,SH40001,RT30672,HUB_NAKHCHIVAN,HUB_KHACHMAZ,2025-09-14,22:30:00,11.944,12.74,0.937520,10,0,0.0,0,DR573,flat,OR925651|OR923180|OR935502|OR935463|OR929429,market_estimate,Sheki,Absheron,700.8,745.3,123.96
2,SH40002,RT31922,HUB_KHANKENDI,HUB_KALBAJAR,2021-10-31,22:30:00,15.225,16.33,0.932333,9,0,0.0,1,DR612,flat,OR933071|OR932503,market_estimate,Khachmaz,Ganja,651.1,753.1,87.25


**3.2 — Building the departure timestamp and converting hub names to regions**

In [6]:

df['departure_dt'] = pd.to_datetime(df['departure_date'] + ' ' + df['departure_time'])
df['region'] = df['origin_hub'].str.replace('HUB_', '', regex=False).str.title()

# handle any naming mismatches between hub names and region names
region_map = {r.title(): r for r in weather['region'].unique()}
df['region'] = df['region'].map(lambda r: region_map.get(r, r))

print(df['region'].value_counts())


region
Lankaran      1247
Qazakh        1243
Khankendi     1227
Kalbajar      1224
Nakhchivan    1216
Ganja         1205
Yevlakh       1203
Khachmaz      1174
Absheron      1120
Sheki         1116
Name: count, dtype: int64


**3.3 — Joining weather data** (`weather` is daily → join by region + nearest date using `merge_asof`)

In [7]:

weather_sorted = weather.copy()
weather_sorted['timestamp'] = pd.to_datetime(weather_sorted['timestamp']).dt.tz_localize(None)
weather_sorted = weather_sorted.sort_values('timestamp')

df = df.sort_values('departure_dt')

merged_parts = []
for region, grp in df.groupby('region'):
    w_region = weather_sorted[weather_sorted['region'] == region].sort_values('timestamp')
    grp = grp.sort_values('departure_dt')
    orig_index = grp.index
    if w_region.empty:
        grp = grp.copy()
        grp['rainfall'] = np.nan
        grp['wind_speed'] = np.nan
        grp['temperature'] = np.nan
    else:
        grp = pd.merge_asof(
            grp.reset_index(drop=True), w_region[['timestamp', 'temperature', 'rainfall', 'wind_speed']],
            left_on='departure_dt', right_on='timestamp', direction='nearest'
        ).drop(columns='timestamp')
        grp.index = orig_index
    merged_parts.append(grp)

df = pd.concat(merged_parts).sort_index()
print('After joining weather:', df.shape)
df[['region', 'departure_dt', 'rainfall', 'wind_speed', 'temperature']].head(3)


After joining weather: (11975, 27)


,region,departure_dt,rainfall,wind_speed,temperature
0,Sheki,2025-01-16 23:30:00,0.0,18.777029,6.295
1,Nakhchivan,2025-09-14 22:30:00,0.0,23.320250,29.300
2,Khankendi,2021-10-31 22:30:00,0.1,9.199390,19.122


**3.4 — Joining traffic data** (region + nearest timestamp using `merge_asof`)

In [8]:

traffic_sorted = traffic.copy()
traffic_sorted['timestamp'] = pd.to_datetime(traffic_sorted['timestamp'])

merged_parts = []
for region, grp in df.groupby('region'):
    t_region = traffic_sorted[traffic_sorted['region'] == region].sort_values('timestamp')
    grp = grp.sort_values('departure_dt')
    orig_index = grp.index
    if t_region.empty:
        grp = grp.copy()
        grp['traffic_congestion_level'] = np.nan
    else:
        grp = pd.merge_asof(
            grp.reset_index(drop=True), t_region[['timestamp', 'traffic_congestion_level']],
            left_on='departure_dt', right_on='timestamp', direction='nearest'
        ).drop(columns='timestamp')
        grp.index = orig_index
    merged_parts.append(grp)

df = pd.concat(merged_parts).sort_index()
print('After joining traffic:', df.shape)
df[['region', 'departure_dt', 'traffic_congestion_level']].head(3)


After joining traffic: (11975, 28)


,region,departure_dt,traffic_congestion_level
0,Sheki,2025-01-16 23:30:00,1.7087
1,Nakhchivan,2025-09-14 22:30:00,3.1952
2,Khankendi,2021-10-31 22:30:00,3.6313


In [9]:

print('Missing (NaN) values:')
print(df[['rainfall', 'wind_speed', 'temperature', 'traffic_congestion_level']].isna().sum())

# fill any remaining NaNs with the region median
for c in ['rainfall', 'wind_speed', 'temperature', 'traffic_congestion_level']:
    df[c] = df.groupby('region')[c].transform(lambda s: s.fillna(s.median()))
    df[c] = df[c].fillna(df[c].median())


Missing (NaN) values:
rainfall                    0
wind_speed                  0
temperature                 0
traffic_congestion_level    0
dtype: int64


## 4. Feature engineering

In [10]:

df['departure_hour'] = df['departure_dt'].dt.hour
df['day_of_week'] = df['departure_dt'].dt.dayofweek          # 0 = Monday
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['is_night'] = df['departure_hour'].isin(list(range(22, 24)) + list(range(0, 6))).astype(int)

# route_difficulty one-hot (flat / mountainous)
df['is_mountainous'] = (df['route_difficulty'] == 'mountainous').astype(int)

feature_cols = [
    'distance_km',
    'is_mountainous',
    'departure_hour',
    'day_of_week',
    'is_weekend',
    'is_night',
    'rainfall',
    'wind_speed',
    'temperature',
    'traffic_congestion_level',
    'actual_load_ton',
    'capacity_ton',
    'utilization_rate',
    'num_packages',
    'is_spot_rental',
]

target_col = 'duration_minutes'

extra_cols = ['shipment_id', 'route_id', 'start_location', 'end_location', 'departure_dt']
model_cols = feature_cols + [target_col] + [c for c in extra_cols if c not in feature_cols]
model_df = df[model_cols].dropna()
print('Final dataset for modeling:', model_df.shape)
model_df[feature_cols + [target_col]].describe().T


Final dataset for modeling: (11975, 21)


,count,mean,std,min,25%,50%,75%,max
distance_km,11975.0,430.204752,214.788402,55.200000,244.100000,426.300000,616.850000,799.70000
is_mountainous,11975.0,0.299040,0.457856,0.000000,0.000000,0.000000,1.000000,1.00000
departure_hour,11975.0,10.144134,10.214099,0.000000,2.000000,3.000000,22.000000,23.00000
day_of_week,11975.0,2.980960,1.999199,0.000000,1.000000,3.000000,5.000000,6.00000
is_weekend,11975.0,0.285428,0.451637,0.000000,0.000000,0.000000,1.000000,1.00000
is_night,11975.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.00000
rainfall,11975.0,1.962656,5.758021,0.000000,0.000000,0.000000,1.100000,144.49998
wind_speed,11975.0,13.797505,7.104219,2.052316,8.707238,11.734564,16.899881,57.06766
temperature,11975.0,18.249144,9.695782,-7.948000,10.351000,17.950000,25.859500,41.95000
traffic_congestion_level,11975.0,1.841564,1.171466,0.200000,0.943300,1.587100,2.491800,5.00000


## 5. Train / Test split

In [11]:

X = model_df[feature_cols]
y = model_df[target_col]

X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X, y, model_df[['shipment_id', 'route_id', 'start_location', 'end_location', 'departure_dt', 'distance_km']],
    test_size=0.2, random_state=42
)

print('Train:', X_train.shape, ' Test:', X_test.shape)


Train: (9580, 15)  Test: (2395, 15)


## 6. Baseline — `distance_km / avg_speed`

Speed = historical average speed (`distance_km / duration_minutes`), computed only on **train** to avoid data leakage.


In [12]:

avg_speed = (X_train['distance_km'] / y_train).mean()   # km/min
print(f'Historical average speed: {avg_speed:.4f} km/min  (~{avg_speed*60:.1f} km/h)')

baseline_pred_train = X_train['distance_km'] / avg_speed
baseline_pred_test  = X_test['distance_km'] / avg_speed

baseline_rmse_train = mean_squared_error(y_train, baseline_pred_train) ** 0.5
baseline_rmse_test  = mean_squared_error(y_test, baseline_pred_test) ** 0.5
baseline_mae_train  = mean_absolute_error(y_train, baseline_pred_train)
baseline_mae_test   = mean_absolute_error(y_test, baseline_pred_test)
baseline_r2_train   = r2_score(y_train, baseline_pred_train)
baseline_r2_test    = r2_score(y_test, baseline_pred_test)

print(f'Baseline RMSE (train/test): {baseline_rmse_train:.2f} / {baseline_rmse_test:.2f}')
print(f'Baseline MAE  (train/test): {baseline_mae_train:.2f} / {baseline_mae_test:.2f}')
print(f'Baseline R2   (train/test): {baseline_r2_train:.4f} / {baseline_r2_test:.4f}')


Historical average speed: 0.9605 km/min  (~57.6 km/h)
Baseline RMSE (train/test): 54.90 / 56.45
Baseline MAE  (train/test): 42.24 / 43.55
Baseline R2   (train/test): 0.9449 / 0.9426


## 7. Comparing 6 different models

We train 6 regressors with reasonable default settings and compare them on **train** and **test** RMSE, MAE, and R2.

1. Linear Regression
2. Ridge Regression
3. Random Forest Regressor
4. Gradient Boosting Regressor
5. Extra Trees Regressor
6. XGBoost Regressor


In [13]:
candidate_models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(random_state=42),
    'RandomForest': RandomForestRegressor(random_state=42),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'ExtraTrees': ExtraTreesRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42),
}

results = []
fitted_models = {}

for name, model in candidate_models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    results.append({
        'Model': name,
        'RMSE_train': mean_squared_error(y_train, pred_train) ** 0.5,
        'RMSE_test': mean_squared_error(y_test, pred_test) ** 0.5,
        'MAE_train': mean_absolute_error(y_train, pred_train),
        'MAE_test': mean_absolute_error(y_test, pred_test),
        'R2_train': r2_score(y_train, pred_train),
        'R2_test': r2_score(y_test, pred_test),
    })

comparison_df = pd.DataFrame(results).sort_values('R2_test', ascending=False).reset_index(drop=True)
comparison_df

,Model,RMSE_train,RMSE_test,MAE_train,MAE_test,R2_train,R2_test
0,RandomForest,1.912638e+01,53.023982,1.425144e+01,40.169534,0.993309,0.949363
1,ExtraTrees,7.003345e-13,54.572753,5.266865e-13,41.264691,1.000000,0.946362
2,GradientBoosting,5.060053e+01,54.787684,3.897038e+01,42.246054,0.953166,0.945939
3,LinearRegression,5.441305e+01,56.215069,4.202448e+01,43.450894,0.945843,0.943085
4,Ridge,5.441311e+01,56.216830,4.202458e+01,43.452545,0.945843,0.943081
5,XGBoost,2.882111e+01,57.980328,2.177373e+01,43.842803,0.984806,0.939454


**Best model (by Test R2):**

In [14]:
best_model_name = comparison_df.sort_values('R2_test', ascending=False).iloc[0]['Model']
best_model_row  = comparison_df[comparison_df['Model'] == best_model_name].iloc[0]

print(f"Best model (Test R2) : {best_model_name}")
print(f"  Test RMSE : {best_model_row['RMSE_test']:.2f} min")
print(f"  Test MAE  : {best_model_row['MAE_test']:.2f} min")
print(f"  Test R2   : {best_model_row['R2_test']:.4f}")

Best model (Test R2) : RandomForest
  Test RMSE : 53.02 min
  Test MAE  : 40.17 min
  Test R2   : 0.9494


In [15]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,          # instead of unlimited — main overfit lever
    min_samples_leaf=3,    # forces leaves to have enough samples, smooths predictions
    max_features='sqrt',
    max_samples=0.8,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

pred_train = rf_model.predict(X_train)
pred_test = rf_model.predict(X_test)

print("R2 train:", r2_score(y_train, pred_train))
print("R2 test:", r2_score(y_test, pred_test))

R2 train: 0.929932994796687
R2 test: 0.9054616673802962


### Actually, max_depth is too high, although it's the best param for the model, however we took max_depth as 13 and checked the results

In [16]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=13,
    min_samples_split=6,
    min_samples_leaf=2,
    max_features=0.7,
    max_samples=0.8,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

pred_train = rf_model.predict(X_train)
pred_test = rf_model.predict(X_test)

In [17]:
r2_train = r2_score(y_train, pred_train)
r2_test = r2_score(y_test, pred_test)
gap = r2_train - r2_test

print("R2 train:", r2_train)
print("R2 test:", r2_test)
print("Gap:", gap) # less than 0.05

final_results = [{
    'Model': 'Optimized_RandomForest',
    'RMSE_train': mean_squared_error(y_train, pred_train) ** 0.5,
    'RMSE_test': mean_squared_error(y_test, pred_test) ** 0.5,
    'MAE_train': mean_absolute_error(y_train, pred_train),
    'MAE_test': mean_absolute_error(y_test, pred_test),
    'R2_train': r2_score(y_train, pred_train),
    'R2_test': r2_score(y_test, pred_test),
}]

final_df = pd.DataFrame(final_results)
final_df

R2 train: 0.9757527081634682
R2 test: 0.9456171836444098
Gap: 0.03013552451905832


,Model,RMSE_train,RMSE_test,MAE_train,MAE_test,R2_train,R2_test
0,Optimized_RandomForest,36.408785,54.950292,27.563768,42.021845,0.975753,0.945617


## 8. Save Best (Tuned) Model with joblib

In [18]:
import joblib, os

os.makedirs("models", exist_ok=True)
model_path = "models/target4_RandomForest_tuned.joblib"
joblib.dump(rf_model, model_path)

print(f"Saved tuned model to: {model_path}")

Saved tuned model to: models/target4_RandomForest_tuned.joblib


## 9. 7-Day ETA Forecast (by Route)

There is no calendar-indexed history per route (each shipment is a one-off event), so a recursive day-by-day forecast like Targets 1-3 isn't possible here. Instead, for each route we freeze its historical average conditions (distance, weather, traffic, load, departure hour) and vary only the calendar-dependent features (`day_of_week`, `is_weekend`, `is_night`) across the next 7 days — since future weather/traffic for a specific date isn't knowable ahead of time, this is a **typical-week ETA outlook** per route, not a true weather-aware forecast.

In [19]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

FORECAST_DAYS = 7
last_date_t4 = df['departure_dt'].max().normalize()
future_dates_t4 = pd.date_range(last_date_t4 + pd.Timedelta(days=1), periods=FORECAST_DAYS, freq='D')

route_profile = model_df.groupby('route_id').agg({
    'distance_km': 'mean',
    'is_mountainous': 'first',
    'rainfall': 'mean',
    'wind_speed': 'mean',
    'temperature': 'mean',
    'traffic_congestion_level': 'mean',
    'actual_load_ton': 'mean',
    'capacity_ton': 'mean',
    'utilization_rate': 'mean',
    'num_packages': 'mean',
    'is_spot_rental': lambda x: x.mode().iloc[0] if len(x.mode()) else 0,
    'departure_hour': lambda x: int(x.mode().iloc[0]) if len(x.mode()) else 12,
    'start_location': 'first',
    'end_location': 'first',
}).reset_index()

# Build ALL (route x day) rows first, then predict once in a single batch call
rows = []
meta = []
for _, route in route_profile.iterrows():
    is_night = int(route['departure_hour'] in list(range(22, 24)) + list(range(0, 6)))
    for fdate in future_dates_t4:
        rows.append({
            'distance_km': route['distance_km'],
            'is_mountainous': route['is_mountainous'],
            'departure_hour': route['departure_hour'],
            'day_of_week': fdate.dayofweek,
            'is_weekend': int(fdate.dayofweek >= 5),
            'is_night': is_night,
            'rainfall': route['rainfall'],
            'wind_speed': route['wind_speed'],
            'temperature': route['temperature'],
            'traffic_congestion_level': route['traffic_congestion_level'],
            'actual_load_ton': route['actual_load_ton'],
            'capacity_ton': route['capacity_ton'],
            'utilization_rate': route['utilization_rate'],
            'num_packages': route['num_packages'],
            'is_spot_rental': route['is_spot_rental'],
        })
        meta.append({
            'route_id': route['route_id'],
            'start_location': route['start_location'],
            'end_location': route['end_location'],
            'date': fdate.date(),
            'day_of_week': fdate.day_name(),
        })

X_batch = pd.DataFrame(rows)[feature_cols]
preds = rf_model.predict(X_batch)              # ONE call for everything, no loop-spam
preds = [round(float(max(0.0, p)), 1) for p in preds]

forecast_df_t4 = pd.DataFrame(meta)
forecast_df_t4['predicted_duration_minutes'] = preds

pivot_t4 = forecast_df_t4.pivot_table(
    index=['route_id', 'start_location', 'end_location'],
    columns='date', values='predicted_duration_minutes'
)

print(f"7-day ETA outlook ({future_dates_t4[0].date()} to {future_dates_t4[-1].date()}), model: RandomForest (tuned)")
pivot_t4

7-day ETA outlook (2026-06-13 to 2026-06-19), model: RandomForest (tuned)


,,date,2026-06-13,2026-06-14,2026-06-15,2026-06-16,2026-06-17,2026-06-18,2026-06-19
route_id,start_location,end_location,,,,,,,
RT30000,Qazakh,Khachmaz,708.6,709.1,708.1,710.4,709.6,708.7,708.5
RT30001,Lankaran,Ganja,777.7,778.7,773.2,774.3,774.6,774.4,776.0
RT30002,Yevlakh,Lankaran,922.4,927.1,909.4,914.6,917.3,918.6,919.9
RT30003,Khankendi,Lankaran,147.5,147.6,149.4,149.0,148.7,148.2,148.2
RT30004,Sheki,Khankendi,194.9,194.6,197.0,196.3,196.0,195.5,195.1
...,...,...,...,...,...,...,...,...,...
RT32994,Khankendi,Nakhchivan,496.4,497.5,494.7,495.4,495.3,495.5,496.7
RT32995,Lankaran,Absheron,583.5,585.7,592.5,589.2,585.9,584.7,583.0
RT32996,Nakhchivan,Yevlakh,498.8,499.2,496.9,497.8,497.5,497.6,499.7


### 7-day ETA forecast — JSON output

In [20]:
import json

json_records_t4 = []
for route_id, grp in forecast_df_t4.groupby('route_id'):
    grp = grp.sort_values('date')
    first = grp.iloc[0]
    json_records_t4.append({
        "route_id": route_id,
        "start_location": first['start_location'],
        "end_location": first['end_location'],
        "model": "RandomForest_tuned",
        "forecast": [
            {"date": str(r['date']), "day_of_week": r['day_of_week'], "predicted_duration_minutes": float(r['predicted_duration_minutes'])}
            for _, r in grp.iterrows()
        ],
    })

forecast_json_t4 = json.dumps(json_records_t4, indent=2, ensure_ascii=False)

with open("target4_7day_forecast.json", "w", encoding="utf-8") as f:
    f.write(forecast_json_t4)

print(f"Saved 7-day ETA forecast for {len(json_records_t4)} routes to target4_7day_forecast.json")
print(forecast_json_t4[:800])

Saved 7-day ETA forecast for 2943 routes to target4_7day_forecast.json
[
  {
    "route_id": "RT30000",
    "start_location": "Qazakh",
    "end_location": "Khachmaz",
    "model": "RandomForest_tuned",
    "forecast": [
      {
        "date": "2026-06-13",
        "day_of_week": "Saturday",
        "predicted_duration_minutes": 708.6
      },
      {
        "date": "2026-06-14",
        "day_of_week": "Sunday",
        "predicted_duration_minutes": 709.1
      },
      {
        "date": "2026-06-15",
        "day_of_week": "Monday",
        "predicted_duration_minutes": 708.1
      },
      {
        "date": "2026-06-16",
        "day_of_week": "Tuesday",
        "predicted_duration_minutes": 710.4
      },
      {
        "date": "2026-06-17",
        "day_of_week": "Wednesday",
        "predicted_duration_minutes": 709.6
      },
      {
        "date": 
